# Week 0 — Dataset tour: follow one purchase order

Procure-to-pay in one line: **purchase order** (what we agreed to buy, at what price) → **goods receipt** (what arrived) → **supplier invoice** (what we are asked to pay) → **three-way match** (do all three agree?) → **payment**.

In this tour you follow **PO-1001** through those documents with plain SQL. Harness 1 will do the same thing automatically from Week 3 on.

In [ ]:
# Setup (run first)
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "course-tools @ git+https://github.com/harness-km/harness-starter@v1#subdirectory=course-tools"],
                   check=True)
    from google.colab import drive
    drive.mount("/content/drive")

import sqlite3
import pandas as pd
import course_tools as ct
from course_tools.helpers import show_pdf

env = ct.bootstrap()
con = sqlite3.connect(env.db)
q = lambda sql, *args: pd.read_sql_query(sql, con, params=args)

## 1. Two companies, one group

Nilgiri Distributors is one group with two companies. Every purchase order, and so every invoice, belongs to one of them, and each pays in its own currency.

In [ ]:
q("select entity_id, name, address, currency, tax_id_type, tax_id from entities")

## 2. The supplier master

Fourteen suppliers: six in the US and six in India, plus one in Germany and one in Canada that sell to the US company. Note three things:

- **currency:** the currency each supplier bills in. The purchase order is in the same currency.
- **tax ID:** an EIN for US suppliers, a GSTIN for Indian ones (15 characters; the first two digits are the state code).
- **bank details:** an invoice is only paid to the account held here, never to one printed on an invoice.

In [ ]:
q("select supplier_id, name, country, currency, tax_id_type, tax_id, bank_account, bank_code from suppliers")

## 3. What we ordered: PO-1001

In [ ]:
display(q("select po_id, supplier_id, entity_id, currency, po_date from purchase_orders where po_id = ?", "PO-1001"))
q("""select l.line_no, l.sku_id, s.description, l.qty, l.unit_price, s.tax_rate
     from po_lines l join skus s using (sku_id) where l.po_id = ?""", "PO-1001")

## 4. What arrived: the goods receipt

In [ ]:
q("""select r.grn_id, r.received_date, g.po_line_no, g.sku_id, g.qty_received
     from goods_receipts r join grn_lines g using (grn_id) where r.po_id = ?""", "PO-1001")

## 5. What we are asked to pay: invoice INV-A

Predict before you look: if everything ordered arrived at the PO price, what should the invoice total be? The US company buys these goods to resell, and has a resale certificate on file with the supplier, so no sales tax is charged. (Packaging it buys for its own use is taxed, at the Dallas, Texas rate of 8.25%.)

In [ ]:
show_pdf(env.invoices / "INV-A.pdf")

## 6. Past invoices

Invoices already processed and paid. The duplicate checks in Week 6 compare new invoices with these.

In [ ]:
q("select invoice_id, supplier_id, invoice_number, invoice_date, po_id, currency, total, status from invoices limit 10")

## Your turn

Pick another purchase order from the catalogue and repeat steps 3–5. Try one from each company: the PO on `INV-11` (US, dollars) and the PO on `INV-10` (India, rupees, with GST). What is different about each? Write one sentence per invoice in your notes; you will meet them again in Week 6.

In [ ]:
pd.read_csv(env.data / "catalogue.csv")[["id", "case", "po_reference", "currency", "total", "note"]]